# PBL6: HUẤN LUYỆN & KIỂM ĐỊNH PHOWHISPER VỚI KIẾN TRÚC LORA MỞ RỘNG (CROSS-ATTENTION & FFN)
### Đề tài: Nền tảng Đánh giá Kỹ năng Thuyết trình Tự động Đa Phương thức (ĐHBK Đà Nẵng - DUT)
**AI & Data Lead:** Nguyễn Đỗ Thắng | **GVHD:** TS. Lê Minh Trí

---
### ĐIỂM NÂNG CẤP KHOA HỌC & KIẾN TRÚC HỌC MÁY (ARCHITECTURAL ENHANCEMENT):
1. **Chuẩn hóa Giọng Chuẩn Miền Nam / Phổ thông (VIVOS + Thuyết trình chuẩn):** Loại bỏ các mẫu phương ngữ miền Trung nặng (Nghệ An, Hà Tĩnh, Quảng Bình, Quảng Trị, Huế) gây loãng gradient và Underfitting; chuẩn hóa bằng tập ngữ liệu chuẩn quốc gia **VIVOS** kết hợp dữ liệu thuyết trình có từ đệm.
2. **Can thiệp Kiến trúc Extended Cross-Attention LoRA:** Mở rộng `target_modules` sang các tầng **Decoder Cross-Attention (`encoder_attn`: `q_proj`, `k_proj`, `v_proj`, `out_proj`)** và **Feed-Forward (`fc1`, `fc2`)** giúp Decoder tiếp nhận trực tiếp tín hiệu ngập ngừng âm học từ Encoder.
3. **Hàm Mất mát Nhạy Từ đệm (`FillerAwareSeq2SeqTrainer`):** Nhân trọng số phạt $\gamma_{\text{filler}} = 1.5$ tại các vị trí token từ đệm (*ờ, à, ừm, kiểu như*) để chống lại thiên kiến tự xóa từ đệm của Whisper gốc.
4. **Đánh giá Kép Không Rò rỉ Người nói (Speaker-Disjoint):** Đo đồng thời **WER, CER** và **Word-boundary Filler Event Precision / Recall / F1**; tối ưu VRAM chạy mượt mà 100% trên **Google Colab T4 15GB VRAM** (`gradient_checkpointing=True`, `fp16=True`, `torch.cuda.empty_cache()`).

In [ ]:
# =============================================================================
# BƯỚC 1: KẾT NỐI GOOGLE DRIVE & CÀI ĐẶT MÔI TRƯỜNG CHUẨN COLAB T4 (15GB VRAM)
# =============================================================================
import os
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/PBL6_Weights'
os.makedirs(DRIVE_DIR, exist_ok=True)

!pip uninstall -y torchao > /dev/null 2>&1
!pip install -q transformers datasets evaluate peft accelerate soundfile jiwer librosa
!nvidia-smi
print(f'\n[OK] Đã kết nối Drive tại: {DRIVE_DIR}')

In [ ]:
# =============================================================================
# BƯỚC 2: GIẢI NÉN DATASET THUYẾT TRÌNH (HOẶC TỰ ĐỘNG KHỞI TẠO CÙNG VIVOS)
# =============================================================================
import os, glob, zipfile

extract_dir = '/content/dut_speech_16k'
os.makedirs(extract_dir, exist_ok=True)

search_paths = [
    '/content/*.zip',
    '/content/drive/MyDrive/*.zip',
    '/content/drive/MyDrive/PBL6*/*.zip',
    '/content/drive/MyDrive/*dut*/*.zip'
]
found_zips = []
for sp in search_paths:
    found_zips.extend(glob.glob(sp))

if found_zips:
    target_zip = found_zips[0]
    print(f'[*] Đang giải nén {target_zip} -> {extract_dir}...')
    with zipfile.ZipFile(target_zip, 'r') as zf:
        zf.extractall(extract_dir)
    print('[OK] Giải nén hoàn tất!')
else:
    print('[!] Không tìm thấy file zip local, hệ thống sẽ tự động tải VIVOS miền Nam chuẩn ở Bước 4.')

In [ ]:
# =============================================================================
# BƯỚC 3: BỘ ĐO TỪ ĐỆM RANH GIỚI TỪ (WORD-BOUNDARY) & LỌC GIỌNG MIỀN NAM/PHỔ THÔNG
# =============================================================================
import re, glob, os
import pandas as pd
from datasets import Dataset, Audio, concatenate_datasets, load_dataset

NONLEXICAL_FILLERS = ['ờm', 'ừm', 'ờ', 'à', 'ừ', 'ư']
DISCOURSE_FILLERS = ['kiểu như là', 'kiểu như', 'thì là', 'nói chung là']
CENTRAL_HEAVY_DIALECT_KEYWORDS = ['nghe_an', 'ha_tinh', 'quang_binh', 'quang_tri', 'tp_hue']

def detect_filler_events_in_text(text: str):
    if not text:
        return [], 0, 0
    clean_str = re.sub(r'[,\.\!\?\;\:\"\(\)\[\]\-]+', ' ', text.lower())
    clean_str = re.sub(r'\s+', ' ', clean_str).strip()
    events, occupied = [], []
    nonlex_cnt, disc_cnt = 0, 0
    for phrase in DISCOURSE_FILLERS:
        pat = r'(?:^|(?<=\s))' + re.escape(phrase) + r'(?=\s|$)'
        for m in re.finditer(pat, clean_str):
            sp = (m.start(), m.end())
            if not any(not (sp[1] <= s[0] or sp[0] >= s[1]) for s in occupied):
                occupied.append(sp)
                events.append({'surface': phrase, 'type': 'discourse_filler'})
                disc_cnt += 1
    for word in NONLEXICAL_FILLERS:
        pat = r'(?:^|(?<=\s))' + re.escape(word) + r'(?=\s|$)'
        for m in re.finditer(pat, clean_str):
            sp = (m.start(), m.end())
            if not any(not (sp[1] <= s[0] or sp[0] >= s[1]) for s in occupied):
                occupied.append(sp)
                events.append({'surface': word, 'type': 'nonlexical_filler'})
                nonlex_cnt += 1
    return events, nonlex_cnt, disc_cnt

def compute_filler_f1_corpus(predictions, references):
    tp, fp, fn = 0, 0, 0
    for pred, ref in zip(predictions, references):
        p_evs, _, _ = detect_filler_events_in_text(pred)
        r_evs, _, _ = detect_filler_events_in_text(ref)
        p_list = [e['surface'] for e in p_evs]
        r_list = [e['surface'] for e in r_evs]
        matched = [False] * len(r_list)
        for p_item in p_list:
            found = False
            for r_i, r_item in enumerate(r_list):
                if not matched[r_i] and p_item == r_item:
                    matched[r_i] = True
                    tp += 1
                    found = True
                    break
            if not found:
                fp += 1
        fn += sum(1 for m in matched if not m)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 1.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 1.0
    f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
    return {'filler_precision': round(prec * 100, 2), 'filler_recall': round(rec * 100, 2), 'filler_f1': round(f1 * 100, 2)}

csv_candidates = glob.glob(f'{extract_dir}/**/metadata.csv', recursive=True)
if csv_candidates:
    csv_path = csv_candidates[0]
    data_dir = os.path.dirname(csv_path)
    df_dut = pd.read_csv(csv_path)
    df_dut['audio'] = df_dut['audio_filename'].apply(lambda fn: os.path.join(data_dir, fn))
    df_dut = df_dut[df_dut['audio'].apply(os.path.exists)].reset_index(drop=True)

    CANONICAL_MAP = {
        'nguoi_03_son_dang_nghe_an': 'spk_03_son_dang',
        'nguoi_10_son_dang_pho_thong': 'spk_03_son_dang'
    }
    df_dut['canonical_speaker_id'] = df_dut['speaker_id'].apply(lambda s: CANONICAL_MAP.get(s, s))

    # Lọc bỏ phương ngữ miền Trung nặng theo góp ý của GVHD để tránh Underfitting
    mask_central = df_dut['speaker_id'].apply(lambda s: any(kw in str(s).lower() for kw in CENTRAL_HEAVY_DIALECT_KEYWORDS))
    df_filtered = df_dut[~mask_central].reset_index(drop=True)
    if len(df_filtered) >= 25:
        print(f'[+] Đã loại bỏ {len(df_dut) - len(df_filtered)} mẫu phương ngữ miền Trung nặng -> Giữ {len(df_filtered)} mẫu Giọng Phổ thông/Miền Nam.')
        df_dut = df_filtered

    unique_spks = sorted(df_dut['canonical_speaker_id'].unique())
    test_spk = 'spk_08_hong_trang' if 'spk_08_hong_trang' in unique_spks else unique_spks[-1]
    rem_spks = [s for s in unique_spks if s != test_spk]
    dev_spk = rem_spks[-1] if len(rem_spks) >= 2 else test_spk
    df_dut['split'] = df_dut['canonical_speaker_id'].apply(lambda s: 'test' if s == test_spk else ('dev' if s == dev_spk else 'train'))

    df_train = df_dut[df_dut['split'] == 'train'].reset_index(drop=True)
    df_dev = df_dut[df_dut['split'] == 'dev'].reset_index(drop=True)
    df_test = df_dut[df_dut['split'] == 'test'].reset_index(drop=True)
    print(f'[SPEAKER SPLIT] Train: {len(df_train)} mẫu | Dev: {len(df_dev)} mẫu | Test: {len(df_test)} mẫu')
else:
    df_train, df_dev, df_test = None, None, None

In [ ]:
# =============================================================================
# BƯỚC 4: NẠP PROCESSOR & CHUẨN HÓA DỮ LIỆU VIVOS MIỀN NAM + THUYẾT TRÌNH
# =============================================================================
import torch
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor, WhisperForConditionalGeneration

MODEL_NAME = 'vinai/PhoWhisper-small'
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, language='Vietnamese', task='transcribe')
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language='Vietnamese', task='transcribe')

try:
    vivos = load_dataset('vivos')
except Exception:
    vivos = load_dataset('ademax/vivos-vie-speech2text')
vivos = vivos.cast_column('audio', Audio(sampling_rate=16000))

if df_train is not None and len(df_train) > 0:
    dut_train_ds = Dataset.from_pandas(df_train[['audio', 'transcription']]).cast_column('audio', Audio(sampling_rate=16000))
    dut_dev_ds = Dataset.from_pandas(df_dev[['audio', 'transcription']]).cast_column('audio', Audio(sampling_rate=16000))
    dut_test_ds = Dataset.from_pandas(df_test[['audio', 'transcription']]).cast_column('audio', Audio(sampling_rate=16000))
    n_vivos = min(250, len(vivos['train']))
    vivos_train = vivos['train'].select(range(n_vivos)).map(lambda b: {'transcription': b.get('sentence', b.get('transcription', ''))})
    train_dataset = concatenate_datasets([dut_train_ds, vivos_train]).shuffle(seed=42)
    eval_dataset = dut_dev_ds
    test_dataset = dut_test_ds
else:
    train_dataset = vivos['train'].select(range(min(400, len(vivos['train'])))).map(lambda b: {'transcription': b.get('sentence', b.get('transcription', ''))})
    eval_dataset = vivos['test'].select(range(min(60, len(vivos['test'])))).map(lambda b: {'transcription': b.get('sentence', b.get('transcription', ''))})
    test_dataset = vivos['test'].select(range(60, min(120, len(vivos['test'])))).map(lambda b: {'transcription': b.get('sentence', b.get('transcription', ''))})

def prepare_batch(batch):
    audio = batch['audio']
    batch['input_features'] = feature_extractor(audio['array'], sampling_rate=audio['sampling_rate']).input_features[0]
    batch['labels'] = tokenizer(batch['transcription']).input_ids
    return batch

train_dataset = train_dataset.map(prepare_batch, remove_columns=train_dataset.column_names, num_proc=2)
eval_dataset = eval_dataset.map(prepare_batch, remove_columns=eval_dataset.column_names, num_proc=2)
test_dataset = test_dataset.map(prepare_batch, remove_columns=test_dataset.column_names, num_proc=2)
print(f'[OK] Đã tiền xử lý Log-Mel (VIVOS Miền Nam + Thuyết trình)! Train: {len(train_dataset)} | Dev: {len(eval_dataset)} | Test: {len(test_dataset)}')

In [ ]:
# =============================================================================
# BƯỚC 5: ĐO BASELINE A0 (PHOWHISPER-SMALL GỐC) & GIẢI PHÓNG VRAM T4
# =============================================================================
import gc
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int
    def __call__(self, features: List[Dict[str, Union[List[int], Any]]]) -> Dict[str, Any]:
        input_features = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')
        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch

metric_wer = evaluate.load('wer')
metric_cer = evaluate.load('cer')

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * metric_wer.compute(predictions=pred_str, references=label_str)
    cer = 100 * metric_cer.compute(predictions=pred_str, references=label_str)
    f_stats = compute_filler_f1_corpus(pred_str, label_str)
    return {'wer': round(wer, 2), 'cer': round(cer, 2), **f_stats}

base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
base_model.config.forced_decoder_ids = None
base_model.config.suppress_tokens = []
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor, decoder_start_token_id=base_model.config.decoder_start_token_id)

baseline_trainer = Seq2SeqTrainer(
    model=base_model,
    args=Seq2SeqTrainingArguments(
        output_dir='/content/tmp_a0',
        per_device_eval_batch_size=4,
        predict_with_generate=True,
        generation_max_length=128,
        fp16=True,
        report_to='none'
    ),
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor
)
a0_test_metrics = baseline_trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix='a0_test')
print('[KẾT QUẢ BASELINE A0 TRÊN TẬP TEST ĐỘC LẬP]:', a0_test_metrics)

del baseline_trainer
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# =============================================================================
# BƯỚC 6: CẢI TIẾN KIẾN TRÚC EXTENDED CROSS-ATTENTION LORA & FILLER-AWARE LOSS
# =============================================================================
# Nguyên lý Học máy Chuyên sâu (Architectural Enhancement):
# 1. Mở rộng LoRA sang toàn bộ tầng Cross-Attention của Decoder (encoder_attn:
#    q_proj, k_proj, v_proj, out_proj) để Decoder tiếp nhận trực tiếp đặc trưng
#    âm học ngập ngừng (F0 phẳng, năng lượng thấp) từ Encoder truyền sang.
# 2. Bổ sung LoRA vào tầng chiếu FFN (fc1, fc2) để tăng dung lượng phân tách
#    từ đệm đơn ('ờ', 'à', 'ừm') khỏi âm tiết trùng âm trong từ ghép tiếng Việt.
# 3. Thiết kế FillerAwareSeq2SeqTrainer nhân trọng số phạt (gamma=1.5) tại các
#    vị trí token từ đệm để chống lại hiện tượng tự động xóa từ đệm của Whisper.
# =============================================================================
import torch.nn as nn
from peft import LoraConfig, get_peft_model
import json, shutil

class FillerAwareSeq2SeqTrainer(Seq2SeqTrainer):
    def __init__(self, *args, filler_token_ids=None, filler_weight=1.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.filler_token_ids = filler_token_ids or set()
        self.filler_weight = float(filler_weight)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        if labels is not None and self.filler_weight > 1.0 and len(self.filler_token_ids) > 0:
            loss_fct = nn.CrossEntropyLoss(reduction='none', ignore_index=-100)
            per_token_loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1)).view(labels.size())
            weights = torch.ones_like(per_token_loss)
            for fid in self.filler_token_ids:
                weights = torch.where(labels == fid, self.filler_weight, weights)
            valid_mask = (labels != -100).float()
            weighted_loss = (per_token_loss * weights * valid_mask).sum() / (valid_mask.sum() + 1e-8)
            return (weighted_loss, outputs) if return_outputs else weighted_loss
        return (outputs.loss, outputs) if return_outputs else outputs.loss

filler_token_ids = set()
for phrase in NONLEXICAL_FILLERS + DISCOURSE_FILLERS:
    for variant in [phrase, f' {phrase}']:
        for tid in tokenizer.encode(variant, add_special_tokens=False):
            filler_token_ids.add(tid)

EXTENDED_TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'out_proj', 'fc1', 'fc2']
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=EXTENDED_TARGET_MODULES,
    lora_dropout=0.05,
    bias='none'
)
base_model.config.use_cache = False
if hasattr(base_model, 'enable_input_require_grads'):
    base_model.enable_input_require_grads()

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

CKPT_DIR = os.path.join(DRIVE_DIR, 'checkpoints_phowhisper_extended_lora')
FINAL_LORA_DIR = os.path.join(DRIVE_DIR, 'whisper_lora_vietnamese')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(FINAL_LORA_DIR, exist_ok=True)

# Cấu hình chống tràn VRAM tuyệt đối trên Google Colab T4 15GB (Batch=2, GradAccum=8, Gradient Checkpointing)
training_args = Seq2SeqTrainingArguments(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    weight_decay=0.01,
    max_grad_norm=1.0,
    warmup_steps=20,
    num_train_epochs=8,
    eval_strategy='epoch',
    save_strategy='epoch',
    predict_with_generate=True,
    generation_max_length=128,
    fp16=True,
    gradient_checkpointing=True,
    per_device_eval_batch_size=4,
    logging_steps=10,
    remove_unused_columns=False,
    label_names=['labels'],
    load_best_model_at_end=True,
    metric_for_best_model='filler_f1',
    greater_is_better=True,
    save_total_limit=2,
    report_to='none'
)

trainer = FillerAwareSeq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    filler_token_ids=filler_token_ids,
    filler_weight=1.5
)

trainer.train()

a3_test_metrics = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix='a3_test')
print('\n================================================================================')
print('   KẾT QUẢ ĐỐI CHIẾU ABLATION TRƯỚC & SAU KHI NÂNG CẤP EXTENDED CROSS-ATTN LORA')
print('================================================================================')
print(f'  • A0 (PhoWhisper gốc)          : WER = {a0_test_metrics["a0_test_wer"]}% | CER = {a0_test_metrics["a0_test_cer"]}% | Filler F1 = {a0_test_metrics["a0_test_filler_f1"]}%')
print(f'  • A3 (Extended Cross-Attn LoRA): WER = {a3_test_metrics["a3_test_wer"]}% | CER = {a3_test_metrics["a3_test_cer"]}% | Filler F1 = {a3_test_metrics["a3_test_filler_f1"]}%')

model.save_pretrained(FINAL_LORA_DIR)
processor.save_pretrained(FINAL_LORA_DIR)
with open(os.path.join(FINAL_LORA_DIR, 'experiment_a0_vs_a3_report.json'), 'w', encoding='utf-8') as f:
    json.dump({
        'architecture': 'PhoWhisper-small + Extended Cross-Attention & FFN LoRA',
        'target_modules': EXTENDED_TARGET_MODULES,
        'filler_loss_weight': 1.5,
        'a0_baseline': a0_test_metrics,
        'a3_extended_lora': a3_test_metrics
    }, f, indent=2, ensure_ascii=False)

zip_out = shutil.make_archive('/content/whisper_lora_vietnamese', 'zip', FINAL_LORA_DIR)
shutil.copy2(zip_out, os.path.join(DRIVE_DIR, 'whisper_lora_vietnamese.zip'))
print(f'\n[SUCCESS] Đã lưu trọn bộ trọng số Extended LoRA chuẩn vinai/PhoWhisper-small tại: {FINAL_LORA_DIR}')